In [ ]:
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
from scoring_templates import SCORING_TEMPLATES, assign_score_tier, score_user_with_breakdown

In [ ]:
# Main function manual scoring

def run_manual_scoring_pipeline(json_data, classification_model_path, desired_profile="Other", rental_context="relaxed"):
    # 1. Parse JSON transactions
    df = pd.json_normalize(json_data)
    df['userId'] = df['userId'].astype(str).str.strip()
    df['description'] = df['description'].fillna('unknown').astype(str).str.strip()
    df['merchantName'] = df['merchantName'].fillna('unknown').astype(str).str.strip()
    df['merchantCode'] = df['merchantCode'].fillna('unknown').astype(str).str.strip()
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce').fillna(0.0)
    df['type'] = df['type'].fillna('unknown').astype(str).str.strip().str.lower()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # 2. Run classification
    X_desc = df[['description', 'merchantName', 'merchantCode']].fillna('').agg(' '.join, axis=1)
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    X_embeddings = encoder.encode(X_desc.tolist(), show_progress_bar=True)
    clf_model = joblib.load(classification_model_path)
    df['predicted_subcategory'] = clf_model.predict(X_embeddings)

    # 3. Recurrence detection
    df = smart_detect_recurrence(df)

    # 4. Feature engineering
    df_features = feature_engineering_with_forecasts(df)

    # 5. Scoring
    template = SCORING_TEMPLATES.get((desired_profile, rental_context), SCORING_TEMPLATES[("Other", "relaxed")])
    all_breakdowns, scores, tiers = [], [], []
    for _, row in df_features.iterrows():
        final_score, breakdown = score_user_with_breakdown(row, template)
        scores.append(final_score)
        tiers.append(assign_score_tier(final_score))
        for b in breakdown:
            b['userId'] = row['userId']
        all_breakdowns.extend(breakdown)

    df_features['financial_behavior_score'] = scores
    df_features['risk_tier'] = tiers
    df_breakdown = pd.DataFrame(all_breakdowns)

    # 6. Save results
    df_features.to_csv("user_scores_with_forecasts_final.csv", index=False)
    df_breakdown.to_csv("user_score_breakdowns_final.csv", index=False)

    print("\n✅ Final manual scoring + ARIMA forecasts + breakdowns saved successfully!")
    display(df_features.head())
    display(df_breakdown.head())



In [ ]:
# Main function model scoring

def run_model_scoring_pipeline(json_data, classification_model_path, risk_model_path):
    # 1. Parse JSON transactions
    df = pd.json_normalize(json_data)
    df['userId'] = df['userId'].astype(str).str.strip()
    df['description'] = df['description'].fillna('unknown').astype(str).str.strip()
    df['merchantName'] = df['merchantName'].fillna('unknown').astype(str).str.strip()
    df['merchantCode'] = df['merchantCode'].fillna('unknown').astype(str).str.strip()
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce').fillna(0.0)
    df['type'] = df['type'].fillna('unknown').astype(str).str.strip().str.lower()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # 2. Run classification
    X_desc = df[['description', 'merchantName', 'merchantCode']].fillna('').agg(' '.join, axis=1)
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    X_embeddings = encoder.encode(X_desc.tolist(), show_progress_bar=True)
    clf_model = joblib.load(classification_model_path)
    df['predicted_subcategory'] = clf_model.predict(X_embeddings)

    # 3. Recurrence detection
    df = smart_detect_recurrence(df)

    # 4. Feature engineering
    df_features = feature_engineering_with_forecasts(df)

    # 5. Risk scoring
    risk_model = joblib.load(risk_model_path)
    X_risk = df_features.drop(columns=['userId'])
    df_features['default_probability'] = risk_model.predict_proba(X_risk)[:, 1]
    df_features['risk_tier'] = pd.cut(
        df_features['default_probability'],
        bins=[0, 0.3, 0.6, 0.8, 1.0],
        labels=["Green - Very Low Risk", "Yellow - Medium Risk", "Orange - High Risk", "Red - Very High Risk"]
    )

    # 6. Save results
    df_features.to_csv("user_scores_final.csv", index=False)

    print("\n✅ Final model scoring + forecasts saved successfully!")
    display(df_features.head())
